In [1]:
import xarray as xr
import os
import pandas as pd
import matplotlib.pyplot as plt
from dateutil.parser import ParserError
import cycler
import matplotlib.ticker as ticker
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import seaborn as sns
import matplotlib.gridspec as gridspec
from typing import List

In [2]:
# Results directory
results_directory = '../../Raven_EPSGO_hydrographs 1/'

# Start and end years for the future climate period
start_year= 2015
end_year= 2100

# Start and end years for historical climate period
hist_start= 1981
hist_end= 2015

# Output directory for plots
output_directory= './outputs/'

In [3]:
def raven_process_average_annual_ensemble(results_directory: str, 
                     region: str,                      
                     scenario: str, 
                     subbasin_id: str,                     
                     start_year: int, 
                     end_year: int) -> xr.DataArray:
    """
    Processes ensemble data for a specified subbasin, scenario, and variable, 
    filtering by start and end year, and calculates average annual runoff.

    Parameters:
        results_directory (str): Directory containing the ensemble files.
        region (str): milk or st mary
        scenario (str): Scenario identifier to filter the files.
        subbasin_id (str): Subbasin ID to filter the files.
        start_year (int): Start year for filtering data.
        end_year (int): End year for filtering data.

    Returns:
        xr.DataArray: Processed average annual runoff shifted to start on October 1st.
    """
    # List of member names to exclude
    exclude_members = [
        "CanESM5", "CNRM-CM6-1", "EC-Earth3", "IPSL-CM6A-LR", "UKESM1-0-LL",
        "NorESM2-MM", "EC-Earth3-CC", "NESM3", "EC-Earth3-Veg", "INM-CM4-8", "TaiESM1"
    ]
    
    # Check for files of required subbasin and scenario
    file_list = [os.path.join(results_directory, filename) 
                 for filename in os.listdir(results_directory) 
                 if (region in filename) and (scenario in filename)]             
    
    # Create an empty list to store xarray DataArrays
    data_arrays: List[xr.DataArray] = []

    # Loop through each file
    for file_path in file_list:
        # Skip files with excluded member names
        if any(member in file_path for member in exclude_members):
            continue
        
        # Read the file into a pandas DataFrame
        df = pd.read_csv(file_path,  index_col=1, low_memory=False)

        # Find the matching column
        matching_cols = [col for col in df.columns if subbasin_id in col and ('observed' not in col.lower())]
        
        # Keep only the matching column(s)
        df = df[matching_cols]

        # Convert the DataFrame to an xarray DataArray
        data_array = xr.DataArray(df.values, dims=('date', 'subbasin'), coords={'date': df.index, 'subbasin': df.columns})

        # Append the DataArray to the list
        data_arrays.append(data_array)

    # Concatenate the list of DataArrays along a new dimension 'member'
    ds = xr.concat(data_arrays, dim='member')

    # Convert 'DATE' coordinate to datetime, specifying the format and handling errors
    ds['date'] = pd.to_datetime(ds['date'], format='%Y-%m-%d', errors='coerce')

    # Convert 'member' coordinate to float
    ds = ds.astype(float)

    # Sort the dataset along the 'DATE' coordinate
    ds = ds.sortby('date')

    # Filter the sorted dataset to include only data between the specified years
    ds_filtered = ds.sel(date=(ds['date'].dt.year >= start_year) & (ds['date'].dt.year <= end_year))

    # Calculate the mean of these daily means for each member to get the average annual runoff
    average_annual_runoff = ds_filtered.groupby('date.dayofyear').mean(dim='date')

    # Drop leap day (dayofyear = 366)
    average_annual_runoff = average_annual_runoff.sel(dayofyear=~(average_annual_runoff['dayofyear'] == 366))
    
    # Shift data to start on October 1st
    dayofyear_shifted = ((average_annual_runoff['dayofyear'] - 274) % 365) + 1

    # Update the 'dayofyear' coordinate with the shifted values
    average_annual_runoff['dayofyear'] = dayofyear_shifted

    # Sort the data_array along the 'dayofyear' coordinate
    average_annual_runoff_shifted = average_annual_runoff.sortby('dayofyear')

    # Drop the subbasin dimension since it will always be 1
    average_annual_runoff_shifted = average_annual_runoff_shifted.squeeze(dim="subbasin")
    
    return average_annual_runoff_shifted

In [4]:
def TtCoM(data):
    """Compute the Time to Center of Mass (TtCoM)"""
    days = np.arange(1, 366)  # Days 1 to 365
    total_flow = np.sum(data)
    
    if total_flow > 0:
        return np.sum(days * data) / total_flow
    else:
        return np.nan

def TtP(data):
    """Compute the Time to Peak (TtP)"""
    if isinstance(data, xr.DataArray):  # Check if data is an xarray DataArray
        return int(data.argmax().item()) + 1  # Convert to integer
    else:
        return np.argmax(data) + 1  # Use NumPy's argmax for regular arrays
        
def SpringPulseOnset(data):
    """Compute the onset of the spring pulse"""
    mean_flow = np.mean(data)
    cumulative_sum = np.cumsum(data - mean_flow)
    
    if isinstance(data, xr.DataArray):  
        onset_day = int(cumulative_sum.argmin().item()) + 1  # Ensure we get a raw integer
    else:
        onset_day = np.argmin(cumulative_sum) + 1
    
    return onset_day

In [5]:
def compute_hydrologic_metrics(data, mean_dim=None, historical=None):
    """
    Compute Time to Center of Mass (TtCoM), Time to Peak (TtP), 
    Spring Pulse Onset, and Total Flow Volume for a given DataArray. 
    If historical values are provided, return both the computed values 
    and the difference between the current ensemble and the historical values 
    (future - historical).

    Parameters:
        data (xr.DataArray): Input hydrologic data.
        mean_dim (str, optional): Dimension along which to compute the mean 
                                  before applying metrics.
        historical (list, optional): A list of historical values for comparison 
                                     [TtCoM, TtP, Spring Pulse Onset, Total Volume].

    Returns:
        tuple: 
            - result (list): [TtCoM, TtP, Spring Pulse Onset, Peak Flow, Total Volume] as numeric values.
            - difference (list or None): [Computed - Historical] for each metric if historical is provided, otherwise None.
                                         A negative value indicates the event is occurring earlier.
    """
    if mean_dim:
        data_mean = data.mean(dim=mean_dim)
    else:
        data_mean = data

    # Compute metrics
    ttcom = float(TtCoM(data_mean))  # Time to Center of Mass
    ttp = int(TtP(data_mean))  # Time to Peak
    spring_pulse_onset = int(SpringPulseOnset(data_mean))  # Spring Pulse Onset
    peak_flow = float(data_mean.max().item())  # Peak Flow as maximum value
    total_volume = float(data_mean.sum().item()) * 86400  # Total Volume (converted to volume)

    # Result list
    result = [ttcom, ttp, spring_pulse_onset, peak_flow, total_volume]

    if historical:
        # Compute the difference for each metric
        difference = [
            r - h if i != 4 else ((r - h) / h) * 100  # For volume, compute percentage difference
            for i, (r, h) in enumerate(zip(result, historical))
        ]
        return result, difference

    return result, None

### Read Results

In [6]:
# Create averate annual data for MREIB ssp245
mreib_ssp245= raven_process_average_annual_ensemble(results_directory, 'milk', 'ssp245', '114036', start_year, end_year)

# Create average annual pseudo historical for MREIB ssp245
mreib_hist_ssp245= raven_process_average_annual_ensemble(results_directory, 'milk', 'ssp245', '114036', hist_start, hist_end)

ValueError: Input DataArray is not 1-D.

In [ ]:
# Create averate annual data for MREIB ssp370
mreib_ssp370 = raven_process_average_annual_ensemble(results_directory, 'milk', 'ssp370', '114036', start_year, end_year)

# Create average annual pseudo historical for MREIB ssp370
mreib_hist_ssp370 = raven_process_average_annual_ensemble(results_directory, 'milk', 'ssp370', '114036', hist_start, hist_end)

In [ ]:
# Create averate annual data for SMRIB ssp245
smrib_ssp245 = raven_process_average_annual_ensemble(results_directory, 'stmary', 'ssp245', '113518', start_year, end_year)

# Create average annual pseudo historical for SMRIB ssp245
smrib_hist_ssp245 = raven_process_average_annual_ensemble(results_directory, 'stmary', 'ssp245', '113518', hist_start, hist_end)

In [ ]:
# Create averate annual data for SMRIB ssp370
smrib_ssp370 = raven_process_average_annual_ensemble(results_directory, 'stmary', 'ssp370', '113518', start_year, end_year)

# Create average annual pseudo historical for SMRIB ssp370
smrib_hist_ssp370 = raven_process_average_annual_ensemble(results_directory, 'stmary', 'ssp370', '113518', hist_start, hist_end)

### Delta Plots

In [ ]:
# Define colors for ensembles
future_color1 = 'dodgerblue'
future_color2 = 'mediumslateblue'
historical_color1 = 'mediumseagreen'
historical_color2 = 'goldenrod'
historical_forcing_color = 'red'
historical_obs_color = 'black'  # New color for observed historical

In [ ]:
# Combine the historical ensembles
mreib_hist_combined = xr.concat([mreib_hist_ssp245, mreib_hist_ssp370], dim='member')

plt.figure(figsize=(16, 8))  # Match the layout size

# Compute medians
ens_hist_median = mreib_hist_combined.median(dim='member')
ens1_median = mreib_ssp245.median(dim='member')
ens2_median = mreib_ssp370.median(dim='member')

# Compute stats using the median of the combined historical
historical_stats, empty = compute_hydrologic_metrics(ens_hist_median)
ens1_stats, ens1_diff = compute_hydrologic_metrics(ens1_median, historical=historical_stats)
ens2_stats, ens2_diff = compute_hydrologic_metrics(ens2_median, historical=historical_stats)

# Plot shading for percentile bands
plt.fill_between(
    mreib_hist_combined['dayofyear'],
    mreib_hist_combined.quantile(0.025, dim='member'),
    mreib_hist_combined.quantile(0.975, dim='member'),
    color=historical_color1,
    alpha=0.2
)
plt.fill_between(
    mreib_ssp245['dayofyear'],
    mreib_ssp245.quantile(0.025, dim='member'),
    mreib_ssp245.quantile(0.975, dim='member'),
    color=future_color1,
    alpha=0.2
)
plt.fill_between(
    mreib_ssp370['dayofyear'],
    mreib_ssp370.quantile(0.025, dim='member'),
    mreib_ssp370.quantile(0.975, dim='member'),
    color=future_color2,
    alpha=0.2
)

# Plot median lines
ens_hist_median.plot(color=historical_color1, linestyle='-', label='Historical Combined Median (1981-2015)')
ens1_median.plot(color=future_color1, linestyle='-', label='Future SSP2-4.5 Median (2015-2100)')
ens2_median.plot(color=future_color2, linestyle='-', label='Future SSP3-7.0 Median (2015-2100)')

# Update legend
legend_handles = [
    Patch(facecolor=historical_color1, alpha=0.2, label='Historical Combined (1981–2015) 0.025-0.975 Percentile'),
    Patch(facecolor=future_color1, alpha=0.2, label='Future SSP2-4.5 (2015–2100) 0.025-0.975 Percentile'),
    Patch(facecolor=future_color2, alpha=0.2, label='Future SSP3-7.0 (2015–2100) 0.025-0.975 Percentile'),
    Line2D([0], [0], color=historical_color1, lw=2, linestyle='-', label='Historical Combined Median (1981–2015)'),
    Line2D([0], [0], color=future_color1, lw=2, linestyle='-', label='Future SSP2-4.5 Median (2015–2100)'),
    Line2D([0], [0], color=future_color2, lw=2, linestyle='-', label='Future SSP3-7.0 Median (2015–2100)')
]


plt.legend(handles=legend_handles)

# Update annotations (3 instead of 5)
x_pos = 1.02
y_offset = 0.85

for stats, diff, label, color in [
    (historical_stats, (0, 0, 0, 0, 0), 'Combined Pseudo-Historical (1981–2015):', historical_color1),
    (ens1_stats, ens1_diff, 'Future SSP2-4.5 (2015–2100)', future_color1),
    (ens2_stats, ens2_diff, 'Future SSP3-7.0 (2015–2100)', future_color2)
]:
    ttcom, ttp, spring_pulse_onset, peak_flow, total_volume = stats
    d_ttcom, d_ttp, d_spring_pulse_onset, d_peak_flow, d_total_volume = diff

    annotation = (
        f"{label}\n"
        f"TtCoM: {ttcom:.1f} days (Δ: {d_ttcom:.1f})\n"
        f"TtP: {ttp:.1f} days (Δ: {d_ttp:.1f})\n"
        f"SpringOnset: {spring_pulse_onset:.1f} days (Δ: {d_spring_pulse_onset:.1f})\n"
        f"Peak Flow: {peak_flow:.2e} m3/s (Δ: {d_peak_flow:.2f} %)\n"
        f"Volume: {total_volume:.2e} m3 (% Δ : {d_total_volume:.2f} %)"
    )

    plt.text(
        x_pos, y_offset, annotation, transform=plt.gca().transAxes,
        fontsize=10, verticalalignment='top', horizontalalignment='left',
        bbox=dict(facecolor=color, alpha=0.2, boxstyle='round,pad=1')
    )
    y_offset -= 0.25
    
plt.title(f'Raven Median and Percentile Ranges for Streamflow at MREIB', fontsize=14)
plt.xlabel('Day of Year', fontsize=12)
plt.ylabel('Discharge (m³/s)', fontsize=12)

# Save and show
file_path_allssp = os.path.join(output_directory, f'raven_MREIB_delta_streamflow.png')
plt.tight_layout()
plt.savefig(file_path_allssp)
plt.show()

In [ ]:
# Combine the historical ensembles
smrib_hist_combined = xr.concat([smrib_hist_ssp245, smrib_hist_ssp370], dim='member')

plt.figure(figsize=(16, 8))  # Match the layout size

# Compute medians
ens_hist_median = smrib_hist_combined.median(dim='member')
ens1_median = smrib_ssp245.median(dim='member')
ens2_median = smrib_ssp370.median(dim='member')

# Compute stats using the median of the combined historical
historical_stats, empty = compute_hydrologic_metrics(ens_hist_median)
ens1_stats, ens1_diff = compute_hydrologic_metrics(ens1_median, historical=historical_stats)
ens2_stats, ens2_diff = compute_hydrologic_metrics(ens2_median, historical=historical_stats)

# Plot shading for percentile bands
plt.fill_between(
    smrib_hist_combined['dayofyear'],
    smrib_hist_combined.quantile(0.025, dim='member'),
    smrib_hist_combined.quantile(0.975, dim='member'),
    color=historical_color1,
    alpha=0.2
)
plt.fill_between(
    smrib_ssp245['dayofyear'],
    smrib_ssp245.quantile(0.025, dim='member'),
    smrib_ssp245.quantile(0.975, dim='member'),
    color=future_color1,
    alpha=0.2
)
plt.fill_between(
    smrib_ssp370['dayofyear'],
    smrib_ssp370.quantile(0.025, dim='member'),
    smrib_ssp370.quantile(0.975, dim='member'),
    color=future_color2,
    alpha=0.2
)

# Plot median lines
ens_hist_median.plot(color=historical_color1, linestyle='-', label='Historical Combined Median (1981-2015)')
ens1_median.plot(color=future_color1, linestyle='-', label='Future SSP2-4.5 Median (2015-2100)')
ens2_median.plot(color=future_color2, linestyle='-', label='Future SSP3-7.0 Median (2015-2100)')

# Update legend
legend_handles = [
    Patch(facecolor=historical_color1, alpha=0.2, label='Historical Combined (1981–2015) 0.025-0.975 Percentile'),
    Patch(facecolor=future_color1, alpha=0.2, label='Future SSP2-4.5 (2015–2100) 0.025-0.975 Percentile'),
    Patch(facecolor=future_color2, alpha=0.2, label='Future SSP3-7.0 (2015–2100) 0.025-0.975 Percentile'),
    Line2D([0], [0], color=historical_color1, lw=2, linestyle='-', label='Historical Combined Median (1981–2015)'),
    Line2D([0], [0], color=future_color1, lw=2, linestyle='-', label='Future SSP2-4.5 Median (2015–2100)'),
    Line2D([0], [0], color=future_color2, lw=2, linestyle='-', label='Future SSP3-7.0 Median (2015–2100)')
]

plt.legend(handles=legend_handles)

# Update annotations (3 instead of 5)
x_pos = 1.02
y_offset = 0.85

for stats, diff, label, color in [
    (historical_stats, (0, 0, 0, 0, 0), 'Combined Pseudo-Historical (1981–2015):', historical_color1),
    (ens1_stats, ens1_diff, 'Future SSP2-4.5 (2015–2100)', future_color1),
    (ens2_stats, ens2_diff, 'Future SSP3-7.0 (2015–2100)', future_color2)
]:
    ttcom, ttp, spring_pulse_onset, peak_flow, total_volume = stats
    d_ttcom, d_ttp, d_spring_pulse_onset, d_peak_flow, d_total_volume = diff

    annotation = (
        f"{label}\n"
        f"TtCoM: {ttcom:.1f} days (Δ: {d_ttcom:.1f})\n"
        f"TtP: {ttp:.1f} days (Δ: {d_ttp:.1f})\n"
        f"SpringOnset: {spring_pulse_onset:.1f} days (Δ: {d_spring_pulse_onset:.1f})\n"
        f"Peak Flow: {peak_flow:.2e} m3/s (Δ: {d_peak_flow:.2f} %)\n"
        f"Volume: {total_volume:.2e} m3 (% Δ : {d_total_volume:.2f} %)"
    )

    plt.text(
        x_pos, y_offset, annotation, transform=plt.gca().transAxes,
        fontsize=10, verticalalignment='top', horizontalalignment='left',
        bbox=dict(facecolor=color, alpha=0.2, boxstyle='round,pad=1')
    )
    y_offset -= 0.25
    
plt.title(f'Raven Median and Percentile Ranges for Streamflow at SMRIB', fontsize=14)
plt.xlabel('Day of Year', fontsize=12)
plt.ylabel('Discharge (m³/s)', fontsize=12)

# Save and show
file_path_allssp = os.path.join(output_directory, f'raven_SMRIB_delta_streamflow.png')
plt.tight_layout()
plt.savefig(file_path_allssp)
plt.show()


### Statistics

In [ ]:
# Define metric names
metrics = ["TtCoM", "TtP", "Spring Pulse Onset", 'Peak Flow', "Total Volume"]

# Define units for each metric
units = ["days", "days", "days", 'cms', "m³"]

# Compute historical and future statistics
smrib_pseudo_historical_stats, empty = compute_hydrologic_metrics(smrib_hist_combined.median(dim='member'))
smrib_future_ens1_stats, smrib_future_ens1_diff = compute_hydrologic_metrics(smrib_ssp245.median(dim='member'), historical=smrib_pseudo_historical_stats)
smrib_future_ens2_stats, smrib_future_ens2_diff = compute_hydrologic_metrics(smrib_ssp370.median(dim='member'), historical=smrib_pseudo_historical_stats)

def format_value(val, diff, unit, metric):
    # If diff is zero, don't show the difference part
    if diff == 0:
        if unit == "days":
            return f"{int(val)} {unit}"
        elif unit == "m³":
            return f"{val:.2e} {unit}"  # Keeping in scientific notation for volume
        elif unit == "cms" and metric == "Peak Flow":
            return f"{val:.1f} {unit}"
    else:
        # If diff is non-zero, show the difference
        if unit == "days":
            return f"{int(val)} {unit} (Change of {int(diff)} {unit})"
        elif unit == "m³":
            if metric == "Total Volume":
                return f"{val:.2e} {unit} (Change of {diff:.2f}%)"  # Keeping non-scientific notation
            else:
                return f"{val:.2f} {unit} (Change of {diff:.2f} {unit})"
        elif unit == "cms" and metric == "Peak Flow":
            return f"{val:.1f} {unit} (Change of {diff:.2f} {unit})"

# Updated formatted_values for SMRIB and MREIB
formatted_values = {
    "Baseline (1981-2015)": [format_value(val, 0, unit, metric) for val, unit, metric in zip(smrib_pseudo_historical_stats, units, metrics)],
    "SSP245 (2015-2100)": [format_value(stat, diff, unit, metric) for stat, diff, unit, metric in zip(smrib_future_ens1_stats, smrib_future_ens1_diff, units, metrics)],
    "SSP370 (2015-2100)": [format_value(stat, diff, unit, metric) for stat, diff, unit, metric in zip(smrib_future_ens2_stats, smrib_future_ens2_diff, units, metrics)],
}

# Create DataFrame with formatted values for SMRIB
smrib_statistic_df = pd.DataFrame(formatted_values, index=metrics)

# Create the MultiIndex with both 'SMRIB' and 'MREIB' as the first level
# (This step is done later to avoid the error caused by the 'statistic_df.index' when combining)
mreib_pseudo_historical_stats, _ = compute_hydrologic_metrics(mreib_hist_combined.median(dim='member'))
mreib_future_ens1_stats, mreib_future_ens1_diff = compute_hydrologic_metrics(mreib_ssp245.median(dim='member'), historical=mreib_pseudo_historical_stats)
mreib_future_ens2_stats, mreib_future_ens2_diff = compute_hydrologic_metrics(mreib_ssp370.median(dim='member'), historical=mreib_pseudo_historical_stats)

# Similarly for MREIB
formatted_values_mreib = {
    "Baseline (1981-2015)": [format_value(val, 0, unit, metric) for val, unit, metric in zip(mreib_pseudo_historical_stats, units, metrics)],
    "SSP245 (2015-2100)": [format_value(stat, diff, unit, metric) for stat, diff, unit, metric in zip(mreib_future_ens1_stats, mreib_future_ens1_diff, units, metrics)],
    "SSP370 (2015-2100)": [format_value(stat, diff, unit, metric) for stat, diff, unit, metric in zip(mreib_future_ens2_stats, mreib_future_ens2_diff, units, metrics)],
}

# Create a DataFrame for MREIB with the formatted values
mreib_statistic_df = pd.DataFrame(formatted_values_mreib, index=metrics)

# Now combine the two DataFrames (for SMRIB and MREIB) into one
final_statistic_df = pd.concat([smrib_statistic_df, mreib_statistic_df])

# After combining the two DataFrames, add the MultiIndex
final_statistic_df.index = pd.MultiIndex.from_arrays([
    ['SMRIB'] * len(metrics) + ['MREIB'] * len(metrics),
    final_statistic_df.index
], names=["Region", "Metric"])

# The final DataFrame, 'final_statistic_df', now contains the combined statistics for both SMRIB and MREIB
final_statistic_df

In [ ]:

# Define seasons with day-of-year ranges
seasons = {
    "Fall (Oct-Dec)": (1, 92),
    "Winter (Jan-Mar)": (93, 181),
    "Spring (Apr-Jun)": (182, 273),
    "Summer (Jul-Sep)": (274, 365)
}

# Function to calculate median-based volumes and format output
def calculate_seasonal_volumes(data_245, data_370, data_hist):
    formatted_values = {
        "Baseline Vol (1981-2015)": [],
        "SSP245 Vol (2015-2100)": [],
        "SSP370 Vol (2015-2100)": []
    }
    
    for season_name, (start_day, end_day) in seasons.items():
        # Filter the data for the current season
        season_245 = data_245.sel(dayofyear=slice(start_day, end_day))
        season_370 = data_370.sel(dayofyear=slice(start_day, end_day))
        season_hist = data_hist.sel(dayofyear=slice(start_day, end_day))

        # Compute the median and sum in one step
        volume_245 = season_245.median(dim='member').sum(dim='dayofyear').values
        volume_370 = season_370.median(dim='member').sum(dim='dayofyear').values
        volume_hist = season_hist.median(dim='member').sum(dim='dayofyear').values

        # Compute percentage changes
        change_245 = ((volume_245 - volume_hist) / volume_hist) * 100 if volume_hist != 0 else 0
        change_370 = ((volume_370 - volume_hist) / volume_hist) * 100 if volume_hist != 0 else 0

        # Store formatted values
        formatted_values["Baseline Vol (1981-2015)"].append(f"{volume_hist:.2e} m³")
        formatted_values["SSP245 Vol (2015-2100)"].append(f"{volume_245:.2e} m³ ({change_245:.1f}%)")
        formatted_values["SSP370 Vol (2015-2100)"].append(f"{volume_370:.2e} m³ ({change_370:.1f}%)")

    return pd.DataFrame(formatted_values, index=seasons.keys())

# Create DataFrames for SMRIB and MREIB
smrib_volume_df = calculate_seasonal_volumes(smrib_ssp245, smrib_ssp370, smrib_hist_combined)
mreib_volume_df = calculate_seasonal_volumes(mreib_ssp245, mreib_ssp370, mreib_hist_combined)

# Combine DataFrames with a MultiIndex
final_volume_df = pd.concat([smrib_volume_df, mreib_volume_df])
final_volume_df.index = pd.MultiIndex.from_arrays([
    ['SMRIB'] * len(seasons) + ['MREIB'] * len(seasons),
    final_volume_df.index
], names=["Region", "Season"])
final_volume_df